# OmniBird-Temporal-JEPA on CIFAR10-DVS

**Predict future from past.** Same architecture as the spatial-block xattn (BigBird per-event encoder + LocalCrossAttention pool + tokenizer skip + target centering + symmetric cross-attn pool, no regularizer), but the masking is **temporally causal**:

| | Spatial xattn | Temporal-JEPA (this notebook) |
|---|---|---|
| Patch ordering | Hilbert curve over (x, y, t) | **Sort all events by t**, chop into K=32 contiguous chunks |
| Patches | Hilbert-coherent spatial-temporal blobs | **Temporally-coherent slices** (each patch ≈ "what happened in this ~5ms" across the sensor) |
| Context | Random subset of remaining patches | **First 60% of patches** by time (the past) |
| Target  | 4 blocks of 16 spatial patches | **4 disjoint blocks of 16 contiguous-in-t future patches** |

**Why this is the right pretext task for event cameras:**

1. **Causality.** The context encoder never sees the future. At inference time the trained encoder is a *real-time* model: it predicts what's coming next given only what's already arrived. This matches the sensor's physics.
2. **Temporal dynamics become load-bearing.** To predict a future patch from past patches, the encoder must capture *how* events propagate: motion direction, velocity, polarity flips at moving edges. These are exactly the features event-camera downstream tasks (action recognition, optical flow, depth) need.
3. **No "spatial multi-block" arbitrariness.** Spatial JEPA on events is awkward because event clouds aren't grid-structured. Temporal blocks are the natural decomposition — `t` is a privileged axis with a clear forward direction.
4. **Free multi-view.** Past/future split can be randomized per sample; pretraining sees many different cuts of the same clip.

The architectural pieces (LocalCrossAttention pool, tokenizer skip, target centering, EMA, predictor) transfer unchanged. Only the dataset/masking is different.


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl, glob
sys.path.insert(0, os.path.abspath('..'))
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

from omnibird import (
    OmniBirdConfig,
    BigBirdEventEncoderWithPool, PerceiverPredictor, FixedPosEmbedder,
    TargetCenter, ema_update, make_momentum_schedule,
    jepa_loss, diag_dict, fmt_diag,
    save_atomic, ensure_dir, short_params,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

N_GPUS_REQUESTED = 4
N_GPUS = min(N_GPUS_REQUESTED, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

DATA_ROOT  = "../data/cifar10_dvs_omnibird"
TRAIN_FRAC = 0.8

cfg = OmniBirdConfig()
cfg.coord_dim       = 3                # (x, y, t)
cfg.signal_dim      = 2                # one-hot polarity
cfg.side            = 64
cfg.n_events_total  = 0
cfg.n_events_max    = 8192             # P_max = 256

# Patch layout — patches are TIME-SORTED contiguous chunks of 32 events.
cfg.patch_mode         = True
cfg.patch_size         = 32
cfg.patches_per_block  = 16
cfg.n_pred_blocks      = 4
cfg.n_tgt              = cfg.n_pred_blocks * cfg.patches_per_block   # 64
cfg.ctx_max_patches    = 128                                          # ~80% of past

# Temporal split: first `past_frac` of (time-sorted) patches = past pool
cfg.past_frac          = 0.6

# Position embedding
cfg.pos_embed       = "nerf"
cfg.nerf_freqs      = 5

# Encoder / predictor — same xattn architecture
cfg.d_model          = 256
cfg.n_layers_enc     = 6
cfg.n_heads          = 8
cfg.dim_head         = 32
cfg.d_pred           = 256
cfg.n_layers_pred    = 4
cfg.n_heads_pred     = 6
cfg.dim_head_pred    = 32
cfg.ffn_mult         = 4
cfg.fourier_dim      = 96
cfg.fourier_scale    = 15.0
cfg.predictor_pos_symmetric = True

# BigBird sparse attention
cfg.attention_type   = "bigbird"
cfg.block_size       = 8
cfg.window           = 1
cfg.n_random         = 2
cfg.n_global         = 2
cfg.reinject_pos     = True

# Pool locality bias
cfg.local_bias_scale = 10.0

# JEPA: target centering + per-token LN + smooth_L1 (no regularizer)
cfg.loss_type        = "smooth_l1"
cfg.use_centering    = True
cfg.ema_start        = 0.999
cfg.ema_end          = 0.9999

# Optim
cfg.epochs         = 200
cfg.batch_size     = 64
cfg.lr             = 8e-4
cfg.weight_decay   = 0.05
cfg.warmup_epochs  = 10
cfg.probe_interval = 20
cfg.probe_epochs   = 3
cfg.log_every      = 25
cfg.ckpt_dir       = "./checkpoints_temporal_jepa"
ensure_dir(cfg.ckpt_dir)

P_max = cfg.n_events_max // cfg.patch_size
n_past = int(P_max * cfg.past_frac)
n_future = P_max - n_past
print(f"P_max = {P_max} time-sorted patches  K = {cfg.patch_size}")
print(f"past = {n_past} patches (first {cfg.past_frac:.0%}), future = {n_future} patches")
print(f"context candidates: {n_past}  →  ctx_max = {cfg.ctx_max_patches}")
print(f"target candidates:  {n_future}  →  n_tgt = {cfg.n_tgt}")


## 2. Temporal-JEPA dataset

In [ ]:
class CIFAR10DVSFromClips:
    def __init__(self, root, sensor_hw=(128, 128)):
        self.root = root; self.h, self.w = sensor_hw
        self.clips = sorted(glob.glob(os.path.join(root, "clip_*")))
        if not self.clips:
            raise RuntimeError(f"No clip_* in {root} (resolved: {os.path.abspath(root)})")
    def __len__(self): return len(self.clips)
    def __getitem__(self, idx):
        clip = self.clips[idx]
        ev = np.load(os.path.join(clip, "events_0.npy")).astype(np.float32)
        if ev.shape[0] == 0: ev = np.zeros((1, 4), dtype=np.float32)
        x = ev[:, 0] / max(self.w - 1, 1) * 2.0 - 1.0
        y = ev[:, 1] / max(self.h - 1, 1) * 2.0 - 1.0
        t_raw = ev[:, 2]
        t = (t_raw - t_raw.min()) / max(t_raw.max() - t_raw.min(), 1.0) * 2.0 - 1.0
        p = ev[:, 3]
        if p.max() <= 1.0 and p.min() >= 0.0: p = p * 2.0 - 1.0
        out = np.stack([x, y, t, p], axis=1).astype(np.float32)
        with open(os.path.join(clip, "label_0.txt")) as f: label = int(f.read().strip())
        return out, label


class CIFAR10DVSTemporalJEPADataset(Dataset):
    # Events sorted by t globally, chunked into K consecutive-in-time
    # patches. Context = first past_frac of patches; target = blocks from
    # the future window.

    def __init__(self, base, cfg, train=True):
        self.base = base
        self.cfg  = cfg
        self.train = train
        self.K_total = cfg.n_events_max
        self.K_per   = cfg.patch_size
        self.P       = self.K_total // self.K_per
        self.n_past  = int(self.P * cfg.past_frac)
        self.n_pred_blocks     = cfg.n_pred_blocks
        self.patches_per_block = cfg.patches_per_block
        self.ctx_max = cfg.ctx_max_patches

    def __len__(self): return len(self.base)

    def _build_signal(self, events):
        if self.cfg.signal_dim == 2:
            pol = events[:, 3]
            sig = torch.zeros(events.shape[0], 2, dtype=torch.float32)
            sig[pol > 0, 0] = 1.0
            sig[pol < 0, 1] = 1.0
            return sig
        return events[:, 3:4]

    def __getitem__(self, idx):
        ev_np, label = self.base[idx]
        events = torch.as_tensor(ev_np, dtype=torch.float32)
        N_raw = events.shape[0]

        # Cap to K_total
        N_target = self.K_total
        if N_raw > N_target:
            sel_rng = np.random.RandomState(idx + 7919) if not self.train else np.random
            sel = sel_rng.choice(N_raw, N_target, replace=False)
            events = events[sel]
            N_real = N_target
        else:
            N_real = N_raw

        # Pad to K_total
        if events.shape[0] < N_target:
            pad = torch.zeros(N_target - events.shape[0], 4, dtype=events.dtype)
            events = torch.cat([events, pad], dim=0)
        N = events.shape[0]
        per_event_kpm = torch.zeros(N, dtype=torch.bool)
        per_event_kpm[N_real:] = True            # True at padded

        # GLOBAL SORT BY t. Padded events have t=0 from torch.zeros — assign them
        # +infinity so they end up at the tail of the sorted sequence.
        t = events[:, 2].clone()
        t[per_event_kpm] = float("inf")
        order = torch.argsort(t, stable=True)
        events       = events[order]
        per_event_kpm = per_event_kpm[order]

        # Patchify by chunking K consecutive time-sorted events
        P = self.P; K = self.K_per
        coord_dim = self.cfg.coord_dim          # 3 (x, y, t)
        signal_dim = self.cfg.signal_dim
        coords  = events[:, :3]                 # already normalized
        signal  = self._build_signal(events)
        ev_combined = torch.cat([coords, signal], dim=-1)        # (N, 3+signal_dim)
        patch_events    = ev_combined.view(P, K, 3 + signal_dim)
        patch_centroids = coords.view(P, K, 3).mean(dim=1)        # (P, 3)
        patch_kpm = per_event_kpm.view(P, K).all(dim=1)           # True if all-padding
        event_kpm_per_patch = per_event_kpm.view(P, K)

        # Past = patches 0..n_past-1 (earliest in t).
        # Future = patches n_past..P-1.
        n_past = self.n_past
        past_real   = (~patch_kpm[:n_past]).nonzero(as_tuple=False).squeeze(-1)
        future_real_local = (~patch_kpm[n_past:]).nonzero(as_tuple=False).squeeze(-1)
        future_real = future_real_local + n_past

        if self.train:
            rng = np.random.RandomState()

            # Target: 4 disjoint blocks of patches_per_block consecutive-in-t
            # patches from the future. Try to space them out across the future
            # window so we predict at multiple temporal horizons.
            n_future = len(future_real)
            tgt_blocks = []
            if n_future >= self.patches_per_block:
                # Pick block start positions evenly across the future
                block_starts = np.linspace(
                    0, max(n_future - self.patches_per_block, 0),
                    self.n_pred_blocks, dtype=int)
                for s in block_starts:
                    blk = future_real[s:s + self.patches_per_block].numpy()
                    tgt_blocks.append(blk)
            tgt_idx = np.concatenate(tgt_blocks).astype(np.int64) if tgt_blocks \
                       else np.zeros(0, dtype=np.int64)
            expected_tgt = self.n_pred_blocks * self.patches_per_block
            if len(tgt_idx) < expected_tgt:
                fill_val = int(tgt_idx[-1]) if len(tgt_idx) else 0
                fill = np.full(expected_tgt - len(tgt_idx), fill_val, dtype=np.int64)
                tgt_idx = np.concatenate([tgt_idx, fill]) if len(tgt_idx) else fill

            # Context: sample from past real patches
            past_real_np = past_real.numpy()
            if len(past_real_np) > self.ctx_max:
                ctx_idx = past_real_np[rng.choice(len(past_real_np), self.ctx_max, replace=False)]
                ctx_idx.sort()
            else:
                ctx_idx = past_real_np
            if len(ctx_idx) < self.ctx_max:
                fill_val = int(ctx_idx[0]) if len(ctx_idx) else 0
                fill = np.full(self.ctx_max - len(ctx_idx), fill_val, dtype=ctx_idx.dtype)
                ctx_idx = np.concatenate([ctx_idx, fill])

            tgt_patch_centroids = patch_centroids[torch.from_numpy(tgt_idx)]
            sample = {
                "pool_patch_events":    patch_events,
                "pool_patch_centroids": patch_centroids,
                "pool_patch_kpm":       patch_kpm,
                "pool_event_kpm":       event_kpm_per_patch,
                "ctx_patch_idx":        torch.from_numpy(ctx_idx),
                "tgt_patch_idx":        torch.from_numpy(tgt_idx),
                "tgt_patch_centroids":  tgt_patch_centroids,
                "label":                int(label),
            }
        else:
            # Test path: use all real PAST patches as context (downstream
            # inference is past-only by design).
            real_past_np = past_real.numpy()
            ctx_idx = real_past_np[:self.ctx_max]
            if len(ctx_idx) < self.ctx_max:
                fill_val = int(ctx_idx[0]) if len(ctx_idx) else 0
                fill = np.full(self.ctx_max - len(ctx_idx), fill_val, dtype=ctx_idx.dtype)
                ctx_idx = np.concatenate([ctx_idx, fill])
            sample = {
                "pool_patch_events":    patch_events,
                "pool_patch_centroids": patch_centroids,
                "pool_patch_kpm":       patch_kpm,
                "pool_event_kpm":       event_kpm_per_patch,
                "ctx_patch_idx":        torch.from_numpy(ctx_idx),
                "label":                int(label),
            }
        return {k: v.contiguous().clone() if isinstance(v, torch.Tensor) else v
                for k, v in sample.items()}


base = CIFAR10DVSFromClips(DATA_ROOT)
n = len(base); rng = np.random.RandomState(0)
perm = rng.permutation(n); n_train = int(n * TRAIN_FRAC)
train_idx, test_idx = perm[:n_train], perm[n_train:]
class _Subset:
    def __init__(self, b, i): self.b=b; self.i=i
    def __len__(self): return len(self.i)
    def __getitem__(self, k): return self.b[int(self.i[k])]
base_train = _Subset(base, train_idx); base_test = _Subset(base, test_idx)

train_ds      = CIFAR10DVSTemporalJEPADataset(base_train, cfg, train=True)
train_eval_ds = CIFAR10DVSTemporalJEPADataset(base_train, cfg, train=False)
test_ds       = CIFAR10DVSTemporalJEPADataset(base_test,  cfg, train=False)
train_loader      = DataLoader(train_ds,      batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
train_eval_loader = DataLoader(train_eval_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
test_loader       = DataLoader(test_ds,       batch_size=cfg.batch_size, shuffle=False, num_workers=0)
print(f"train={len(train_ds)}  test={len(test_ds)}")


## 3. Visualize the temporal split

In [ ]:
SAMPLE_IDX = 0
sample = train_ds[SAMPLE_IDX]
classes = ['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
P, K, Fdim = sample["pool_patch_events"].shape
pool_ev   = sample["pool_patch_events"].numpy()
centroids = sample["pool_patch_centroids"].numpy()
patch_kpm = sample["pool_patch_kpm"].numpy()
ev_kpm    = sample["pool_event_kpm"].numpy()
ctx_idx   = sample["ctx_patch_idx"].numpy()
tgt_idx   = sample["tgt_patch_idx"].numpy()
n_past    = int(P * cfg.past_frac)
print(f"sample {SAMPLE_IDX}: label={classes[sample['label']]}  past={n_past} patches  future={P-n_past}")
print(f"real ctx patches: {len(np.unique(ctx_idx))}/{cfg.ctx_max_patches}  "
      f"real tgt patches: {len(np.unique(tgt_idx))}/{cfg.n_tgt}")

events_flat = pool_ev.reshape(P*K, Fdim)
coords = events_flat[:, :3]
pol = events_flat[:, 3] - events_flat[:, 4]
ev_kpm_flat = ev_kpm.reshape(P*K)
real = ~ev_kpm_flat
real_coords = coords[real]
real_pol    = pol[real]
patch_id_per_event = np.repeat(np.arange(P), K)[real]
in_past_event   = patch_id_per_event < n_past
in_future_event = patch_id_per_event >= n_past
in_ctx_event    = np.isin(patch_id_per_event, ctx_idx)
in_tgt_event    = np.isin(patch_id_per_event, tgt_idx)

fig = plt.figure(figsize=(20, 6))

# (a) All events colored by t — shows the global temporal ordering
ax = fig.add_subplot(1, 3, 1, projection='3d')
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=2, c=real_coords[:, 2], cmap='viridis', alpha=0.7)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.set_title(f"(a) all events, color = t  (label={classes[sample['label']]})")

# (b) Past (cool) vs future (warm) split
ax = fig.add_subplot(1, 3, 2, projection='3d')
ax.scatter(real_coords[in_past_event, 0], real_coords[in_past_event, 1], real_coords[in_past_event, 2],
           s=2, c='#3b82f6', alpha=0.5, label=f'past ({in_past_event.sum()} events)')
ax.scatter(real_coords[in_future_event, 0], real_coords[in_future_event, 1], real_coords[in_future_event, 2],
           s=2, c='#f59e0b', alpha=0.5, label=f'future ({in_future_event.sum()} events)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.set_title(f"(b) past/future @ {cfg.past_frac:.0%} time cut")
ax.legend(loc='lower right', fontsize=8)

# (c) Context (red, past) + 4 target blocks (4 colors, future)
ax = fig.add_subplot(1, 3, 3, projection='3d')
ax.scatter(real_coords[~(in_ctx_event | in_tgt_event), 0],
           real_coords[~(in_ctx_event | in_tgt_event), 1],
           real_coords[~(in_ctx_event | in_tgt_event), 2],
           s=1, c='lightgray', alpha=0.2, label='leftover')
ax.scatter(real_coords[in_ctx_event, 0], real_coords[in_ctx_event, 1], real_coords[in_ctx_event, 2],
           s=2, c='#ef4444', alpha=0.45, label=f'context (past)')
TGT_COLORS = ['#fbbf24', '#34d399', '#60a5fa', '#f472b6']
for k in range(cfg.n_pred_blocks):
    block_ids = tgt_idx[k*cfg.patches_per_block:(k+1)*cfg.patches_per_block]
    em = np.isin(patch_id_per_event, block_ids)
    ax.scatter(real_coords[em, 0], real_coords[em, 1], real_coords[em, 2],
               s=6, c=TGT_COLORS[k], alpha=0.7,
               label=f'tgt block {k+1}')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.set_title("(c) context (past, red) → predict 4 future blocks (colors)")
ax.legend(loc='lower right', fontsize=7, ncol=2)
plt.tight_layout(); plt.show()


## 4. Models

In [ ]:
# Shared fixed positional embedder (orthogonal-projected NeRF, frozen)
shared_pos = FixedPosEmbedder(coord_dim=cfg.coord_dim, d_model=cfg.d_model,
                               num_freqs=cfg.nerf_freqs).to(DEVICE)

def make_encoder():
    return BigBirdEventEncoderWithPool(
        d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
        n_heads=cfg.n_heads, dim_head=cfg.dim_head,
        block_size=cfg.block_size, window=cfg.window,
        n_random=cfg.n_random, n_global=cfg.n_global,
        ffn_mult=cfg.ffn_mult,
        signal_dim=cfg.signal_dim, coord_dim=cfg.coord_dim,
        fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
        serial_orders=("z", "z_rev", "hilbert", "hilbert_rev"),
        reinject_pos=cfg.reinject_pos,
        side=cfg.side,
        fixed_pos_embedder=shared_pos,
        local_bias_scale=cfg.local_bias_scale,
    )

context_encoder = make_encoder().to(DEVICE)
target_encoder  = copy.deepcopy(context_encoder).to(DEVICE)
for q in target_encoder.parameters(): q.requires_grad_(False)

predictor = PerceiverPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult, pos_symmetric=cfg.predictor_pos_symmetric,
    pos_embed=cfg.pos_embed, nerf_freqs=cfg.nerf_freqs,
    fixed_pos_embedder=shared_pos,
).to(DEVICE)
target_center = TargetCenter(cfg.d_model, momentum=0.9).to(DEVICE)

if USE_DP:
    device_ids = list(range(N_GPUS))
    context_encoder = nn.DataParallel(context_encoder, device_ids=device_ids)
    target_encoder  = nn.DataParallel(target_encoder,  device_ids=device_ids)
    predictor       = nn.DataParallel(predictor,       device_ids=device_ids)

def _unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m
print(f"context_encoder: {short_params(_unwrap(context_encoder))}")
print(f"predictor      : {short_params(_unwrap(predictor))}")


## 5. Optim + resume

In [ ]:
params = list(_unwrap(context_encoder).parameters()) + list(_unwrap(predictor).parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
warmup_steps = cfg.warmup_epochs * len(train_loader)
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    p = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + math.cos(math.pi * p))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)

LAST = os.path.join(cfg.ckpt_dir, "temporal_jepa_last.pt")
BEST = os.path.join(cfg.ckpt_dir, "temporal_jepa_best.pt")
RESUME = True
history = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
m = cfg.ema_start
if RESUME and os.path.exists(LAST):
    s = torch.load(LAST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    _unwrap(predictor).load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0); best_loss = s.get("best_loss", float("inf"))
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = cfg.ema_end
    print(f"resumed @ ep {s['epoch']}, step {global_step}")
else:
    print("starting fresh.")


## 6. Helpers + probe (probe uses past-only context, matching the causal pretext)

In [ ]:
def gather_ctx_subset(batch):
    pool_ev    = batch["pool_patch_events"].to(DEVICE)
    pool_cen   = batch["pool_patch_centroids"].to(DEVICE)
    pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
    ctx_idx    = batch["ctx_patch_idx"].to(DEVICE)
    B, P, K, Fd = pool_ev.shape
    Pc = ctx_idx.size(1)
    sub_ev    = torch.gather(pool_ev,    1, ctx_idx[..., None, None].expand(B, Pc, K, Fd))
    sub_cen   = torch.gather(pool_cen,   1, ctx_idx[..., None].expand(B, Pc, pool_cen.size(-1)))
    sub_evkpm = torch.gather(pool_evkpm, 1, ctx_idx[..., None].expand(B, Pc, K))
    return sub_ev.reshape(B, Pc*K, Fd), sub_cen, sub_evkpm.reshape(B, Pc*K)


def flatten_pool(batch):
    pool_ev    = batch["pool_patch_events"].to(DEVICE)
    pool_cen   = batch["pool_patch_centroids"].to(DEVICE)
    pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
    B, P, K, Fd = pool_ev.shape
    return pool_ev.view(B, P*K, Fd), pool_cen, pool_evkpm.view(B, P*K)


class LinearProbe(nn.Module):
    def __init__(self, d, n=10):
        super().__init__()
        self.fc = nn.Linear(d, n)
    def forward(self, z): return self.fc(z)


def quick_probe(num_epochs=cfg.probe_epochs, lr=1e-3, wd=1e-4):
    enc = _unwrap(context_encoder)
    saved_rg = [p.requires_grad for p in enc.parameters()]
    for p in enc.parameters(): p.requires_grad_(False)
    enc.eval()
    probe = LinearProbe(cfg.d_model, 10).to(DEVICE)
    opt = AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    ce  = nn.CrossEntropyLoss()
    def _z(b):
        # Probe is given only PAST events (the context subset) — same constraint
        # as the training pretext task. Mean-pool over context centroid features.
        ev, cen, kpm = gather_ctx_subset(b)
        with torch.no_grad():
            g = enc(ev, cen, event_kpm=kpm)              # (B, Pc, D)
        return g.mean(dim=1)
    for _ in range(num_epochs):
        probe.train()
        for b in train_eval_loader:
            y = b["label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            ce(probe(_z(b)), y).backward()
            opt.step()
    probe.eval()
    correct = total = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            correct += (probe(_z(b)).argmax(-1) == y).sum().item(); total += y.size(0)
    for p, rg in zip(enc.parameters(), saved_rg): p.requires_grad_(rg)
    return correct / max(total, 1)


## 7. Training loop — predict future-window features from past-window features

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            tgt_cen = batch["tgt_patch_centroids"].to(DEVICE)

            # Target encoder sees the FULL clip (past + future) — same as
            # i-JEPA's target encoder. Pools at FUTURE centroid positions.
            with torch.no_grad():
                pool_ev, _, pool_kpm = flatten_pool(batch)
                h_tgt_raw = target_encoder(pool_ev, tgt_cen, event_kpm=pool_kpm)
                Dim = h_tgt_raw.size(-1)
                if cfg.use_centering:
                    target_center.update(h_tgt_raw)
                    h_tgt = F.layer_norm(target_center(h_tgt_raw), (Dim,))
                else:
                    h_tgt = F.layer_norm(h_tgt_raw, (Dim,))

            # CONTEXT ENCODER sees ONLY PAST events (the context subset).
            # This is the causality constraint — at inference, the trained
            # encoder is given only past events. Pretext task and inference
            # are now aligned.
            ctx_ev, ctx_cen, ctx_kpm = gather_ctx_subset(batch)
            g_ctx = context_encoder(ctx_ev, ctx_cen, event_kpm=ctx_kpm)

            # Predictor: future-target queries cross-attend over past-context features.
            h_pred = predictor(g_ctx, tgt_cen,
                                ctx_coords=ctx_cen if cfg.predictor_pos_symmetric else None,
                                ctx_key_padding_mask=None)

            loss = jepa_loss(h_pred, h_tgt, loss_type=cfg.loss_type)
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(_unwrap(target_encoder), _unwrap(context_encoder), m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                pred_std = h_pred.std(dim=0).mean().item()
                tgt_std  = h_tgt.std(dim=0).mean().item()
                cos = F.cosine_similarity(h_pred, h_tgt, dim=-1).mean().item()
                print(f"[ep{epoch:02d} st{global_step:05d}]  loss={loss.item():.4f}  "
                      f"pred_std={pred_std:.3f}  tgt_std={tgt_std:.3f}  cos={cos:.3f}  "
                      f"lr={scheduler.get_last_lr()[0]:.1e}  m={m:.5f}")
                history["diag_steps"].append(global_step)
                history["diag_log"].append({"pred_std": pred_std, "tgt_std": tgt_std,
                                              "cos": cos, "loss": loss.item()})

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg
        state = {
            "epoch": epoch, "cfg": cfg.__dict__,
            "context_encoder": _unwrap(context_encoder).state_dict(),
            "target_encoder":  _unwrap(target_encoder).state_dict(),
            "predictor":       _unwrap(predictor).state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss, "history": history,
        }
        save_atomic(state, LAST)
        if improved: save_atomic(state, BEST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  loss={avg:.4f}  m={m:.5f}  "
              f"{time.time()-t0:.1f}s" + ("  *" if improved else ""))

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_p = time.time()
            acc = quick_probe()
            history["probe_accs"].append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_p:.1f}s)")
    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 8. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("smooth_L1"); axes[0].set_title("temporal JEPA loss")
    axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    axes[1].plot(steps, [d["pred_std"] for d in history["diag_log"]], label="pred_std", color="C0")
    axes[1].plot(steps, [d["tgt_std"]  for d in history["diag_log"]], label="tgt_std",  color="C2")
    axes[1].plot(steps, [d["cos"]      for d in history["diag_log"]], label="cos",      color="C3")
    axes[1].axhline(0.01, color='red', ls='--', alpha=0.4)
    axes[1].set_xlabel("step"); axes[1].set_title("diagnostics"); axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3')
    axes[2].set_ylim(0, 100); axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"probe (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()
